In [29]:
import sqlite3

conn = sqlite3.Connection("2025-04-15.db")

cur = conn.cursor()

In [ ]:
import py_vollib.ref_python.black_scholes.greeks.analytical as pyv
import datetime as dt

def getGreeks(date, expiry, stockPrice, r, sigma, strike):
    flag = 'p'
    dateDt = dt.datetime.strptime(date, "%Y-%m-%d")
    expiryDt = dt.datetime.strptime(expiry, "%Y-%m-%d")

    t = (expiryDt-dateDt).days/365.25

    delta = pyv.delta(flag, stockPrice, strike, t, r, sigma)
    gamma = pyv.gamma(flag, stockPrice, strike, t, r, sigma)
    theta = pyv.theta(flag, stockPrice, strike, t, r, sigma)
    vega = pyv.vega(flag, stockPrice, strike, t, r, sigma)
    rho  = pyv.rho(flag, stockPrice, strike, t, r, sigma)
    return delta, gamma, vega, theta, rho

In [25]:
import yfinance as yf
from dateutil.relativedelta import relativedelta

start = dt.datetime(year=2024, month=1, day=1)

yf.Ticker("AAPL").history(start=start, interval="1d").index[0]

Timestamp('2024-01-02 00:00:00-0500', tz='America/New_York')

In [88]:
def quarter_start(start):
    return yf.Ticker("AAPL").history(start=start, interval="1d").index[0]

def quarter_end(start):
    end = start+relativedelta(months=3)
    return yf.Ticker("AAPL").history(end=end, interval="1d").index[-1]

In [59]:
start_date, end_date = quarter_start(start).strftime('%Y-%m-%d'), quarter_end(start).strftime('%Y-%m-%d')
strike = 180
ticker = 'AAPL'

In [41]:
exp_date = cur.execute(f"SELECT DISTINCT(exp) FROM options where ticker='AAPL' AND date = '{start_date}' AND exp >= '{end_date}';").fetchone()
exp_date

('2024-04-19',)

In [74]:
def get_closest_quarterly_exp(start_date, end_date, ticker):
    return cur.execute(f"""SELECT DISTINCT(exp) FROM options
                       where ticker='{ticker}' 
                       AND date = '{start_date}' 
                       AND exp >= '{end_date}';""").fetchone()[0]

def get_closest_strike(start_date, exp, ticker, target_strike):
    return cur.execute(f"""SELECT strike FROM options 
                where ticker='{ticker}' 
                AND date = '{start_date}' 
                AND exp = '{exp}' 
                AND type='P' 
                ORDER BY ABS(strike - {target_strike});""").fetchone()[0]

def get_option_stats(start_date, exp, ticker, strike):
    return cur.execute(f"""SELECT lastPrice, impliedVolatility FROM options 
                where ticker='{ticker}' 
                AND date = '{start_date}' 
                AND exp = '{exp}' 
                AND strike = '{strike}'
                AND type = 'P';""").fetchall()


In [75]:
exp = get_closest_quarterly_exp(start_date, end_date, ticker)

#getGreeks(start_date.strftime("%Y-%m-%d"), exp, 180, 0.05, )

In [76]:
get_option_stats(start_date, exp, 'AAPL', 180)

[(5.4, 0.1997150341796875)]

In [93]:
overall_start = dt.datetime(year=2023, month=7, day=1)

df = yf.Ticker("SPY").history(start=overall_start, interval='1d')[['Open']]

qs = quarter_start(overall_start)
qe = quarter_end(overall_start)

qe

Timestamp('2023-09-29 00:00:00-0400', tz='America/New_York')